# Russian airport disruption report

Cancellations use scheduled Moscow dates and the latest observation of each airport flight movement. Restrictions remain a separate Airports.ru measure.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, HTML

ROOT = Path.cwd()
if not (ROOT / 'tracker.py').exists():
    raise RuntimeError('Start Jupyter from the tracker project directory')
DB = ROOT / 'flights.sqlite3'
con = sqlite3.connect(DB)
airports = pd.read_sql_query(
    'SELECT * FROM airports WHERE active = 1 ORDER BY iata', con
)
obs = pd.read_sql_query('SELECT * FROM flight_observations', con)
obs = obs[obs.airport_iata.isin(airports.iata)].copy()
print(f'Database: {DB}')
print(f'{len(obs):,} observations loaded across {obs.airport_iata.nunique() if len(obs) else 0} airports')


## Daily cancellations by airport

Deduplicated airport flight movements, assigned to their scheduled date in Moscow time. Missing dates are unavailable, not proof of zero cancellations.


In [ ]:
import json
from restriction_report import flight_daily
con.row_factory = sqlite3.Row
config = json.loads((ROOT / 'airports.json').read_text())
flight_counts, missing_times = flight_daily(con.execute('SELECT * FROM flight_observations'), {a['iata'] for a in config})
if flight_counts:
    cancellation_table = pd.DataFrame.from_dict(flight_counts, orient='index').reindex(columns=[a['iata'] for a in config]).sort_index()
    cancellation_table['Total'] = cancellation_table.sum(axis=1, min_count=1)
    cancellation_table.index.name = 'scheduled Moscow date'
    display(cancellation_table)
else:
    display(HTML('<p>No dated flight observations available yet.</p>'))
print(f'{missing_times} observations without usable scheduled times')


## Cancellation trends

The top chart shows the overall trend. The six airport panels use one shared scale so changes can be compared directly.

In [ ]:
from html import escape

if 'cancellation_table' in locals() and not cancellation_table.empty and not cancellation_table.isna().any().any():
    dates = list(cancellation_table.index)
    airport_codes = [c for c in cancellation_table.columns if c != 'Total']
    colors = ['#dc2626', '#2563eb', '#16a34a', '#9333ea', '#ea580c', '#0891b2']
    color_by_airport = dict(zip(airport_codes, colors))
    width, height = 920, 760
    parts = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' style='font:13px sans-serif'>"]
    parts.append("<text x='20' y='28' font-size='20' font-weight='bold'>Russian airport flight cancellations</text>")
    parts.append("<text x='20' y='48' fill='#555'>Latest observations by scheduled Moscow date; gaps are unavailable</text>")

    # Overall trend.
    ox, oy, ow, oh = 70, 75, 790, 170
    total_max = max(float(cancellation_table['Total'].max()), 1)
    parts.append(f"<text x='{ox}' y='{oy - 12}' font-size='16' font-weight='bold'>Total cancellations</text>")
    parts.append(f"<line x1='{ox}' y1='{oy + oh}' x2='{ox + ow}' y2='{oy + oh}' stroke='#555'/><line x1='{ox}' y1='{oy}' x2='{ox}' y2='{oy + oh}' stroke='#555'/>")
    parts.append(f"<text x='12' y='{oy + oh / 2}' transform='rotate(-90 12 {oy + oh / 2})'>cancellations</text>")
    total_points = []
    for j, day in enumerate(dates):
        x = ox + ow * j / max(len(dates) - 1, 1)
        value = float(cancellation_table.loc[day, 'Total'])
        y = oy + oh - oh * value / total_max
        total_points.append((x, y, value, day))
    parts.append(f"<polyline points='{ ' '.join(f'{x:.1f},{y:.1f}' for x, y, _, _ in total_points) }' fill='none' stroke='#111827' stroke-width='4'/>")
    for x, y, value, day in total_points:
        label = pd.Timestamp(day).strftime('%b %d').replace(' 0', ' ')
        parts.append(f"<circle cx='{x:.1f}' cy='{y:.1f}' r='5' fill='#111827'/><text x='{x - 10:.1f}' y='{y - 10:.1f}' font-weight='bold'>{int(value)}</text><text x='{x - 20:.1f}' y='{oy + oh + 22}'>{label}</text>")

    # Airport small multiples.
    max_airport = max(float(cancellation_table[airport_codes].to_numpy().max()), 1)
    panel_w, panel_h = 400, 145
    panel_x = [60, 500]
    panel_y = [285, 455, 625]
    for idx, airport in enumerate(airport_codes):
        px, py = panel_x[idx % 2], panel_y[idx // 2]
        ax, ay, aw, ah = px + 38, py + 28, panel_w - 58, panel_h - 52
        color = color_by_airport[airport]
        parts.append(f"<text x='{px}' y='{py + 15}' font-size='16' font-weight='bold' fill='{color}'>{escape(str(airport))}</text>")
        parts.append(f"<line x1='{ax}' y1='{ay + ah}' x2='{ax + aw}' y2='{ay + ah}' stroke='#999'/><line x1='{ax}' y1='{ay}' x2='{ax}' y2='{ay + ah}' stroke='#999'/>")
        for grid_value in (0, max_airport / 2, max_airport):
            gy = ay + ah - ah * grid_value / max_airport
            parts.append(f"<line x1='{ax}' y1='{gy:.1f}' x2='{ax + aw}' y2='{gy:.1f}' stroke='#e5e7eb'/><text x='{ax - 28}' y='{gy + 4:.1f}'>{int(grid_value)}</text>")
        points = []
        for j, day in enumerate(dates):
            x = ax + aw * j / max(len(dates) - 1, 1)
            value = float(cancellation_table.loc[day, airport]) if pd.notna(cancellation_table.loc[day, airport]) else 0
            y = ay + ah - ah * value / max_airport
            points.append((x, y, value))
        parts.append(f"<polyline points='{ ' '.join(f'{x:.1f},{y:.1f}' for x, y, _ in points) }' fill='none' stroke='{color}' stroke-width='3'/>")
        for j, (x, y, value) in enumerate(points):
            label = pd.Timestamp(dates[j]).strftime('%b %d').replace(' 0', ' ')
            parts.append(f"<circle cx='{x:.1f}' cy='{y:.1f}' r='4' fill='{color}'/><text x='{x - 6:.1f}' y='{y - 9:.1f}'>{int(value)}</text>")
            if idx // 2 == len(panel_y) - 1:
                parts.append(f"<text x='{x - 15:.1f}' y='{ay + ah + 20}'>{label}</text>")
    parts.append('</svg>')
    display(HTML(''.join(parts)))
elif obs.empty:
    display(HTML('<p>No flight observations available yet.</p>'))


## Restrictions and cancellations on matching dates

Select all source airports or only airports.json airports, and select a restriction type. Faded bars identify provisional intervals or incomplete flight collection.


In [ ]:
from restriction_report import dashboard
from html import escape
try:
    html = dashboard(con, config)
    (ROOT / 'dashboard.html').write_text(html, encoding='utf-8')
    display(HTML('<iframe sandbox="allow-scripts" style="width:100%;height:1250px;border:0" srcdoc="' + escape(html, quote=True) + '"></iframe>'))
except sqlite3.OperationalError:
    display(HTML('<p>Run collect_restrictions.py to import restriction history first.</p>'))
